# Pertemuan 2 – Analisis Statistik Data Kualitas Udara Kabupaten Sampang

**Nama:** Raihan Aryanova  
**NIM:** 23XXXXXXXX  
**Program Studi:** Informatika  

## Tujuan Pembelajaran

1. Melakukan migrasi dataset Kualitas Udara ($NO_2$) Kabupaten Sampang dari file CSV lokal ke cloud database Aiven PostgreSQL.
2. Menghubungkan cloud database Aiven PostgreSQL ke KNIME Analytics Platform.
3. Memproses dan mengekstrak nilai statistik deskriptif menggunakan KNIME.
4. Menjelaskan secara teoritis dan matematis seluruh properti statistik yang dihasilkan oleh KNIME.
5. Memberikan contoh perhitungan manual beserta verifikasi menggunakan Python.
6. Menyusun dokumentasi terstruktur untuk dipublikasikan pada web statis Jupyter Book.

## 1. Import Library dan Memuat Dataset

Dataset yang digunakan merupakan hasil pemrosesan data konsentrasi $NO_2$ Kabupaten Sampang periode **24 Agustus 2025 hingga 23 Agustus 2026** (315 hari pengamatan).

In [1]:
import pandas as pd
import numpy as np
import scipy.stats as stats

# Memuat file dataset
file_csv = '../data/no2_sampang.csv'
df = pd.read_csv(file_csv)

print("Jumlah Baris:", len(df))
print("Jumlah Kolom:", len(df.columns))
print("Nama Kolom:")
print(list(df.columns))

df.head()

Jumlah Baris: 315
Jumlah Kolom: 9
Nama Kolom:
['tanggal', 'lat', 'lon', 'NO2', 'status_outlier', 'jenis_outlier', 'perubahan_NO2', 'persentase_perubahan', 'abs_perubahan']


,tanggal,lat,lon,NO2,status_outlier,jenis_outlier,perubahan_NO2,persentase_perubahan,abs_perubahan
0,2025-08-24,-7.2,113.3,0.000022,Normal,Normal,NaN,NaN,NaN
1,2025-08-25,-7.2,113.3,0.000013,Normal,Normal,-9.127703e-06,-40.837802,9.127703e-06
2,2025-08-26,-7.2,113.3,0.000014,Normal,Normal,7.855871e-07,5.940882,7.855871e-07
3,2025-08-27,-7.2,113.3,0.000023,Normal,Normal,9.490909e-06,67.748672,9.490909e-06
4,2025-08-28,-7.2,113.3,0.000021,Normal,Normal,-2.204615e-06,-9.381380,2.204615e-06


## 2. Pemeriksaan Struktur Data dan Missing Value

Pemeriksaan awal dilakukan untuk memverifikasi tipe data dan keberadaan nilai hilang sebelum dimasukkan ke cloud database.

In [2]:
# Menampilkan struktur dan tipe data
df.info()

print("\nJumlah Missing Value pada Setiap Kolom:")
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315 entries, 0 to 314
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   tanggal               315 non-null    object 
 1   lat                   315 non-null    float64
 2   lon                   315 non-null    float64
 3   NO2                   315 non-null    float64
 4   status_outlier        315 non-null    object 
 5   jenis_outlier         315 non-null    object 
 6   perubahan_NO2         314 non-null    float64
 7   persentase_perubahan  314 non-null    float64
 8   abs_perubahan         314 non-null    float64
dtypes: float64(6), object(3)
memory usage: 22.3+ KB

Jumlah Missing Value pada Setiap Kolom:
tanggal                 0
lat                     0
lon                     0
NO2                     0
status_outlier          0
jenis_outlier           0
perubahan_NO2           1
persentase_perubahan    1
abs_perubahan           1
dtype: 

## 3. Penyimpanan Data pada Cloud Database (Aiven PostgreSQL)

Aiven digunakan sebagai penyedia layanan cloud database PostgreSQL agar dataset terpusat dan dapat diakses dari KNIME.

### Tahapan Integrasi Aiven Cloud:
1. Membuat instance PostgreSQL pada platform Aiven.
2. Mencatat parameter koneksi database: *Host*, *Port*, *User*, *Password*, dan *Database Name*.
3. Membuat tabel database `no2_sampang` dengan skema tipe data yang sesuai.
4. Mengunggah seluruh baris data ke cloud database Aiven.

## 4. Penarikan Data Menggunakan KNIME Analytics Platform

Pengambilan data dan perhitungan statistik dilakukan pada KNIME Analytics Platform menggunakan rangkaian node berikut:

1. **CSV Reader**: Membaca file CSV lokal.
2. **PostgreSQL Connector**: Membuka koneksi terenkripsi ke cloud database Aiven PostgreSQL.
3. **DB Writer**: Menuliskan isi CSV ke tabel cloud Aiven secara langsung.
4. **DB Table Selector**: Memilih tabel `no2_sampang` yang berada di cloud Aiven.
5. **DB Reader**: Eksekusi query untuk menarik data dari cloud Aiven ke KNIME Table.
6. **Statistics**: Menghitung dan mengekstrak statistik deskriptif dari setiap kolom.

## 5. Penjelasan Properti Statistik KNIME

- **Column**: Nama kolom atau variabel yang sedang dianalisis.
- **Type**: Tipe data variabel pada KNIME (seperti Number Double, Integer, String, atau Date).
- **Min (Minimum)**: Nilai terkecil dari seluruh observasi pada suatu kolom.
- **Max (Maximum)**: Nilai terbesar dari seluruh observasi pada suatu kolom.
- **Mean**: Nilai rata-rata aritmetika, mengukur titik pusat distribusi data.
- **Median ($Q_2$)**: Nilai tengah data setelah seluruh nilai diurutkan dari terkecil ke terbesar.
- **Std. Dev. (Standard Deviation)**: Simpangan baku sampel ($s$), mengukur rata-rata penyebaran data terhadap nilai rata-ratanya.
- **Variance**: Varians sampel ($s^2$), yaitu kuadrat dari standar deviasi yang menunjukkan besarnya dispersi data.
- **Skewness**: Mengukur tingkat ketidaksimetrisan (kemiringan) distribusi data terhadap rata-ratanya.
- **Kurtosis**: Mengukur tingkat keruncingan puncak distribusi relatif terhadap distribusi normal.
- **IQR (Interquartile Range)**: Jangkauan antarkuartil, yaitu selisih antara Kuartil Ketiga ($Q_3$) dan Kuartil Pertama ($Q_1$).
- **MAD (Median Absolute Deviation)**: Median dari selisih mutlak setiap data terhadap nilai median.
- **No. Missing**: Jumlah baris data yang kosong atau bernilai `NULL`/`NaN`.
- **No. +unlimited**: Jumlah nilai yang bernilai positif tak hingga ($+\infty$).
- **No. -unlimited**: Jumlah nilai yang bernilai negatif tak hingga ($-\infty$).
- **Row Count**: Jumlah total baris atau observasi non-null pada kolom tersebut.

## 6. Formulasi Matematis dan Contoh Perhitungan Statistik

Untuk memahami mekanisme perhitungan setiap properti statistik, digunakan sampel data kecil $X = \{2, 4, 5, 7, 9\}$ dengan jumlah sampel $n = 5$.

### 1. Minimum ($\text{Min}$)
$$\text{Min}(X) = \min(x_1, x_2, \dots, x_n)$$
$$\text{Min}(\{2, 4, 5, 7, 9\}) = 2$$

### 2. Maksimum ($\text{Max}$)
$$\text{Max}(X) = \max(x_1, x_2, \dots, x_n)$$
$$\text{Max}(\{2, 4, 5, 7, 9\}) = 9$$

### 3. Rata-rata ($\bar{x}$ / Mean)
$$\bar{x} = \frac{\sum_{i=1}^{n} x_i}{n}$$
$$\bar{x} = \frac{2 + 4 + 5 + 7 + 9}{5} = \frac{27}{5} = 5.4$$

### 4. Median ($\tilde{x}$)
Data terurut: $2, 4, 5, 7, 9$. Karena $n = 5$ (ganjil), median terletak pada data ke-\frac{5+1}{2} = 3.
$$\tilde{x} = 5$$

### 5. Standar Deviasi Sampel ($s$) dan Varians ($s^2$)
$$s^2 = \frac{\sum_{i=1}^{n} (x_i - \bar{x})^2}{n - 1}$$
$$s^2 = \frac{29.20}{5 - 1} = 7.30$$
$$s = \sqrt{7.30} \approx 2.70185$$

In [3]:
# Verifikasi Perhitungan Menggunakan Python
contoh = np.array([2, 4, 5, 7, 9], dtype=float)

print("Data Sampel:", contoh)
print("Min:", np.min(contoh))
print("Max:", np.max(contoh))
print("Mean:", np.mean(contoh))
print("Median:", np.median(contoh))
print("Varians Sampel:", np.var(contoh, ddof=1))
print("Std. Dev. Sampel:", np.std(contoh, ddof=1))
print("Skewness Sampel:", stats.skew(contoh, bias=False))
print("Kurtosis Sampel (Excess):", stats.kurtosis(contoh, bias=False))
print("MAD:", np.median(np.abs(contoh - np.median(contoh))))

Data Sampel: [2. 4. 5. 7. 9.]
Min: 2.0
Max: 9.0
Mean: 5.4
Median: 5.0
Varians Sampel: 7.3
Std. Dev. Sampel: 2.701851217221259
Skewness Sampel: 0.182523257308998
Kurtosis Sampel (Excess): -0.681178457496717
MAD: 2.0


## 7. Perhitungan Statistik Aktual Dataset Kualitas Udara ($NO_2$)

In [4]:
def hitung_statistik_lengkap(data_frame):
    numeric_cols = data_frame.select_dtypes(include=[np.number]).columns
    hasil = []
    
    for col in numeric_cols:
        s = data_frame[col].dropna()
        n = len(s)
        mean_v = s.mean()
        median_v = s.median()
        min_v = s.min()
        max_v = s.max()
        std_v = s.std(ddof=1) if n > 1 else 0.0
        skew_v = stats.skew(s, bias=False) if n > 2 else np.nan
        kurt_v = stats.kurtosis(s, bias=False) if n > 3 else np.nan
        
        n_missing = data_frame[col].isna().sum()
        n_pos_inf = np.isposinf(data_frame[col]).sum()
        n_neg_inf = np.isneginf(data_frame[col]).sum()
        
        hasil.append({
            'Column': col,
            'Min': min_v,
            'Mean': mean_v,
            'Median': median_v,
            'Max': max_v,
            'Std. Dev.': std_v,
            'Skewness': skew_v,
            'Kurtosis': kurt_v,
            'No. Missing': n_missing,
            'No. +unlimited': n_pos_inf,
            'No. -unlimited': n_neg_inf
        })
    return pd.DataFrame(hasil)

df_stat_aktual = hitung_statistik_lengkap(df)
df_stat_aktual

C:\Users\Raihan aryanova\AppData\Local\Temp\ipykernel_23520\389470251.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew_v = stats.skew(s, bias=False) if n > 2 else np.nan
C:\Users\Raihan aryanova\AppData\Local\Temp\ipykernel_23520\389470251.py:14: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  kurt_v = stats.kurtosis(s, bias=False) if n > 3 else np.nan
C:\Users\Raihan aryanova\AppData\Local\Temp\ipykernel_23520\389470251.py:13: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  skew_v = stats.skew(s, bias=False) if n > 2 else np.nan
C:\Users\Raihan aryanova\AppData\Local\Temp\ipykernel_23520\389470251.py:14: R

,Column,Min,Mean,Median,Max,Std. Dev.,Skewness,Kurtosis,No. Missing,No. +unlimited,No. -unlimited
0,lat,-7.200000e+00,-7.200000e+00,-7.200000e+00,-7.200000,8.895916e-16,NaN,NaN,0,0,0
1,lon,1.133000e+02,1.133000e+02,1.133000e+02,113.300000,1.423347e-14,NaN,NaN,0,0,0
2,NO2,-7.570182e-06,2.121875e-05,2.013691e-05,0.000054,8.698538e-06,0.512415,1.273930,0,0,0
3,perubahan_NO2,-3.718294e-05,-2.368265e-08,-2.369922e-07,0.000040,9.782418e-06,0.105452,2.407859,1,0,0
4,persentase_perubahan,-1.187911e+03,2.893954e-01,-1.724452e+00,349.301921,9.273202e+01,-7.456755,91.182783,1,0,0
5,abs_perubahan,2.689366e-08,7.143639e-06,5.152269e-06,0.000040,6.670956e-06,1.990905,5.111280,1,0,0


## 8. Interpretasi Hasil Statistik dan Kesimpulan

1. **Konsentrasi $NO_2$**: Rata-rata konsentrasi $NO_2$ berada pada angka $2.1219 \times 10^{-5}$ $\text{mol/m}^2$ dengan distribusi miring positif ($0.5124$).
2. **Missing Value**: Hanya terdapat $1$ nilai kosong pada kolom perubahan harian dikarenakan baris pertama tidak memiliki data pembanding sebelumnya.
3. **Nilai Tak Hingga**: Tidak terdapat nilai $+\infty$ maupun $-\infty$ pada seluruh kolom.